# Two-photon preprocessing

Run this notebook before the other four 2P notebooks
(`2p_1_population` ... `2p_4_feedback`): it is the only one that writes data
another notebook reads. It draws no published panel.

**Reads** (`data/2p/`):
- `df_all_by_epoch.pkl` -- raw fluorescence of the 23 L2/3 MFC/LFC sessions;
- `unnormalized_3_quantiles_0.1s_before_sampling_0.1s_after_movement.pkl` and
  `traces_cut_feedback_filtered_full_df.pkl` -- the sampling and feedback epochs
  the choice-selectivity tests run on;
- `normalized_0.1s_before_sampling_0.1s_after_movement.pkl` -- only when the
  max-firing tables are rebuilt.

**Writes** (`data/2p/`, only with `SAVE_DATA`):
- `df_all_by_epoch_df_f_filtered.pkl` -- dF/F of the neurons active during
  sampling (read by `2p_1_population`, `2p_2_sequences`, `2p_3_choice`);
- `data_runs_Unnormalized.pkl`, `data_runs_Feedback_fixed.pkl` -- the
  per-neuron ROC tests, real and shuffled; a run already on disk is reused and
  not rewritten;
- `sgf_all.pkl` -- the significantly tuned neurons;
- `q1_res_dict.pkl`, `q2_res_dict.pkl`, `q3_res_dict.pkl`, `qall_res_dict.pkl`
  -- each neuron's peak position on fast / typical / slow / all trials, written
  only when rebuilt (`LOAD_FROM_PICKLE = False`).

With `SAVE_FIGS` and not `PAPER_FIGURES_ONLY` it also writes the per-neuron
reliability figures under `results/2P/SingleTraces/`.

# Run this to enable loading from relative paths

In [ ]:
%load_ext autoreload
%autoreload 2
if "PKG" not in globals():
  import importlib, sys, pathlib # https://stackoverflow.com/a/50395128/11996983
  PKG = %pwd
  PKG = pathlib.Path(PKG)
  root_parent_level = 1
  root = PKG
  full_pkg = f"{root.name}"
  for _ in range(root_parent_level):
    root = root.parent
    full_pkg = f"{root.name}.{full_pkg}"
    MODULE_PATH = f"{root}{pathlib.os.path.sep}__init__.py"
    MODULE_NAME = f"{root.name}"
    spec = importlib.util.spec_from_file_location(MODULE_NAME, MODULE_PATH)
    module = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = module
    spec.loader.exec_module(module)
  __package__ = full_pkg


# Save plots with no embeded fonts

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']

# Parameters

In [ ]:
# Papermill parameters -- code/run_notebooks.py overrides these.
# Everything is off by default, so running the notebook writes nothing.
SAVE_FIGS = False           # write figures under results/
SAVE_DATA = False           # rewrite cached intermediate data under data/
PAPER_FIGURES_ONLY = False  # skip per-subject / per-session figures

In [ ]:
import pathlib

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from .common.definitions import BrainRegion

PVAL = 0.05
CONCAT_SAMPLING_MOVE = True

fig_save_prefix = "../results/2P/"
pathlib.Path(fig_save_prefix).mkdir(exist_ok=True)

# Load data into a dataframe

In [ ]:
from .twop.dataload import loadEpochTraces

# Raw fluorescence for the 23 M2/ALM L2/3 sessions, cut down to the 30 columns
# this notebook reads. The notebook gives the same accepted neurons and dF/F
# traces on it as on the full 33-session file.
df_all_by_epoch = loadEpochTraces()

# Active neurons extraction: Restrict analysis to neurons that are active during the states we care about

In [ ]:
from .twop.samplingreliability import alignSamplingFrames

# The sampling epoch of every trial, in real time and stretched to a common
# length, z-scored per neuron. Built from the raw df_all_by_epoch, not from the
# dF/F traces.
unnormed_sampling_df, normed_sampling_df, _ = alignSamplingFrames(df_all_by_epoch)

Removing decision time trials and nan-time trials...


Decision time filter: 21,880/22,483 (603 rows (not trials) removed)
Done...
Processing: Split data by trial ['Name', 'Date', 'SessionNum']
Processing: Applying on full trace


  0%|          | 0/23 [00:00<?, ?it/s]

Processing: Applying no normalization to trace
Processing: Applying no normalization to trace
Processing: Applying no normalization to trace
Processing: Applying no normalization to trace
Processing: Applying no normalization to trace
Processing: Applying no normalization to trace
Processing: Applying no normalization to trace
Processing: Applying no normalization to trace
Processing: Applying no normalization to trace
Processing: Applying no normalization to trace
Processing: Applying no normalization to trace
Processing: Applying no normalization to trace
Processing: Applying no normalization to trace
Processing: Applying no normalization to trace
Processing: Applying no normalization to trace
Processing: Applying no normalization to trace
Processing: Applying no normalization to trace
Processing: Applying no normalization to trace
Processing: Applying no normalization to trace
Processing: Applying no normalization to trace
Processing: Applying no normalization to trace
Processing: A

Processing: Aligning traces to 0.1 sec(s) before epoch(s): ['Sampling']  and 0 sec(s) afterwards


Processing: Collapsing the dataframes back into one dataframe
Processing: Aligning traces to 500 sec(s) before epoch(s): ['Movement to Lateral Port']  and 0.1 sec(s) afterwards


... (31 outputs left out)

  0%|          | 0/23 [00:00<?, ?it/s]

Processing: Concatenated epochs of the same trial-number together


  0%|          | 0/5351 [00:00<?, ?it/s]

Processing: Collapsing the dataframes back into one dataframe


In [ ]:
# Sequence characterization. The trace-reliability helpers are shared with
# 2p_4_feedback and tested in twop/tracereliability.py and
# twop/samplingreliability.py.
from .twop.samplingreliability import collectReliability
from .twop.tracereliability import DecideNeurons

np.seterr(all='raise')

_decide_neurons = DecideNeurons(smooth_sigma=1)
all_res_normed_dict, all_res_normed_df = collectReliability(
    normed_sampling_df, _decide_neurons, pdf_str="max_firing_all_normed",
    save_figs=SAVE_FIGS and not PAPER_FIGURES_ONLY, fig_save_prefix=fig_save_prefix)
all_res_unnormed_dict, all_res_unnormed_df = collectReliability(
    unnormed_sampling_df, _decide_neurons, pdf_str="max_firing_all_unnormed",
    save_figs=SAVE_FIGS and not PAPER_FIGURES_ONLY, fig_save_prefix=fig_save_prefix)

## Plot Active neurons distribution

In [ ]:
from .twop.tracereliability import setupEpochAxes

from .common.clr import BrainRegion as BRClr

_MAX_X = None # Will be set to the max trace length
_BINS_STEP = 4
def plotActiveDistSgf(res_dict, isValidAndFirePost, save_fig=False):
    global _MAX_X
    df = pd.DataFrame(res_dict)
    # display(df)
    ex_trace = next(iter(normed_sampling_df.iloc[0].traces_sets["neuronal"].values()))
    # trace_len = len(ex_trace)
    # print("trace_len:", trace_len)
    res_rows = []
    print(ex_trace)
    _MAX_X = len(ex_trace)#/30
    fig, axs,  = plt.subplots(3, 1, figsize=(10, 18))
    fig2, ax_bar = plt.subplots()
    bar_x_ticks = []
    bar_x_tick_labels = []
    for br, br_df in df.groupby("BrainRegion"):
        br = BrainRegion(br)
        br_str = str(br).split("_")[0]
        br_clr = BRClr[BrainRegion(br)]
        count_all = 0
        count_valid = 0
        res_x_valid = []
        res_x_invalid = []
        res_rows_br = []
        for _, row in br_df.iterrows():
            count_all += 1
            row = row.copy()
            is_valid, fire_pos = isValidAndFirePost(row)
            row["valid"]  = is_valid
            res_rows_br.append(row)
            if not is_valid:
                res_x_invalid.append(fire_pos)
                continue
            res_x_valid.append(fire_pos)
            count_valid += 1
            # metric = row.max_vals
            # quantiles = row.quantile_idx
        res_rows += res_rows_br

        res_x_all = np.array(res_x_valid + res_x_invalid)
        assert len(res_x_all) == count_all
        res_x_valid = np.array(res_x_valid)
        assert len(res_x_valid) == count_valid
        res_x_invalid = np.array(res_x_invalid)
        # print("Res x:", res_x)
        bins = np.arange(_MAX_X + _BINS_STEP, step=_BINS_STEP)
        print("bins:", bins)
        #assert res_x_valid.max() <= max_x, f"{res_x_valid.max() = } != {max_x = }"
        for xs, ls, dscrp, count, ax in [(res_x_valid, "-", "Active", count_valid, axs[0]),
                                         (res_x_all, "--", "All", count_all, axs[1]),
                                         (res_x_invalid, ":", "Inactive", count_all-count_valid, axs[2]),
                              ]:
            hist, _ = np.histogram(xs, bins=bins)
            hist = 100*hist.astype(float)/count
            # print("hist:", hist, "- hist sum = ", hist.sum())
            ax.step(bins[:-1], hist,  color=br_clr, where="post", ls=ls,#align="edge", width=1, histtype="step")
                    label=f"{br_str} {dscrp}")
        print(f"{br_str} - All neurons: {count_all:,} - valid: {count_valid:,}"
              f"- Valid prcnt: {100*count_valid/count_all:.2f}")

        # Plot bar
        sess_grpby_len_all = br_df.groupby("ShortName").size()
        # Give it a name so we can merge it
        sess_grpby_len_all.name = "num_all_neurons"
        # display(sess_grpby_len_all)
        br_df_valid = pd.DataFrame(res_rows_br).query("valid")
        sess_grpby_len_valid = br_df_valid.groupby("ShortName").size()
        sess_grpby_len_valid.name = "num_valid_neurons"
        # merge
        sess_grpby = pd.merge(sess_grpby_len_all, sess_grpby_len_valid,
                              left_index=True, right_index=True,)
        # display(sess_grpby)
        sess_grpby["prcnt"] = (
                    100*sess_grpby.num_valid_neurons/sess_grpby.num_all_neurons)
        # display(sess_grpby[["num_active_neurons_df_f", "num_neurons", "prcnt"]])
        prcnt_mean = sess_grpby.prcnt.mean()
        USE_SEM = False
        if USE_SEM:
            prcnt_sem = sess_grpby.prcnt.sem()
            sem_str = "sem"
        else:
            prcnt_sem = sess_grpby.prcnt.std()
            sem_str = "std."
        # ax.errorbar(len(x_ticks), prcnt_mean, yerr=prcnt_sem, fmt="o", color=br_clr)
        ax_bar.bar(len(bar_x_ticks), prcnt_mean, yerr=prcnt_sem, color=br_clr)
        sum_all_neurons = sess_grpby.num_all_neurons.sum()
        sum_all_active = sess_grpby.num_valid_neurons.sum()
        bar_x_tick_labels.append(f"{br_str}\n"
                             #  f"{layer}\n"
                             f"{sum_all_active:,} active / {sum_all_neurons:,} Neurons\n"
                             f"Gloval mean: {100*sum_all_active/sum_all_neurons:.2f}\n"
                             f"Mean: {prcnt_mean:.2f}% ± {prcnt_sem:.2f} {sem_str}\n"
                             )
        bar_x_ticks.append(len(bar_x_ticks))
    ax_bar.set_xticks(bar_x_ticks)
    ax_bar.set_xticklabels(bar_x_tick_labels)
    ax_bar.set_ylabel("Accepted Neurons %")
    ax_bar.set_title("Accepted Neurons % per Brain Region")
    ax_bar.spines[["left", "right", "top"]].set_visible(False)

    for ax, ax_title in zip(axs, ["Active (Accepted)", "All", "Inactive"]):
        ax.set_title(f"{ax_title} Max. Firing Position")
        setupEpochAxes([ax], normed_sampling_df.iloc[0])
        ax.set_ylabel(f"Neurons %")
        for _ax in [ax]:
            _ax.set_xlabel("Normalized Time (s)")
        ax.set_ylim(bottom=0)
        ax.legend(loc="upper center")
    if save_fig:
        fig.savefig(f"{fig_save_prefix}/accepted_neurons_dst.svg")
        fig2.savefig(f"{fig_save_prefix}/accepted_neurons_prcnt.svg")
    plt.show()
    return pd.DataFrame(res_rows)


In [ ]:
PRCNT_VALID = 5
def getIsValidAndFirePostUnnormed(row):
    is_valid = row.prcnt_valid >= PRCNT_VALID
    fire_pos = row.max_idxs.mean() if len(row.max_idxs) else np.nan
    return is_valid, fire_pos

is_active_unormed_df = plotActiveDistSgf(all_res_unnormed_dict, getIsValidAndFirePostUnnormed)

[-0.10978051 -1.0031167   0.1361155   2.4775758   0.4550789   0.807031
 -1.2667735  -0.90999556 -0.28771344 -0.7379092  -0.66570336 -0.47107375
 -1.1482661  -0.09649077 -1.240345   -1.5960062  -0.20068766  0.16150206
 -0.30223718  0.22527222 -0.80020267  0.95700043  0.08415095 -0.5650542
  0.29315862 -0.6777426  -0.67717814 -0.15537755 -0.10195556 -0.31920403
  0.19508372  1.909203   -1.0340394 ]


bins: [ 0  4  8 12 16 20 24 28 32 36]
ALM - All neurons: 1,536 - valid: 609- Valid prcnt: 39.65


bins: [ 0  4  8 12 16 20 24 28 32 36]
M2 - All neurons: 2,340 - valid: 837- Valid prcnt: 35.77


# Neuron filtering and dF/F calculation

In [ ]:
from .pipeline import pipeline, tracesnormalize, tracesmath

def calcDF_F(df, copy_data):
  # df = df[df.anlys_path == df.anlys_path.iloc[0]]
  sampling_rates = df.acq_sampling_rate.unique()
  assert len(sampling_rates) == 1
  assert int(sampling_rates[0]) == 30
  display(df.sole_owner.unique())
  chain = pipeline.Chain(
    pipeline.BySession(),
      pipeline.ApplyFullTrace(
              # sigma is in frames, for the 30 Hz sampling rate asserted above
              tracesmath.GaussianFilter(sigma=1, copy_data=copy_data),
              tracesnormalize.CalcBaseline(track_is_active=False)),
    pipeline.RecombineResults(),
  )
  return chain.run(df)

df_all_by_epoch_df_f = calcDF_F(df_all_by_epoch.copy(), copy_data=True)

array([False])

Processing: Split data by trial ['Name', 'Date', 'SessionNum']
Processing: Applying on full trace


  0%|          | 0/23 [00:00<?, ?it/s]

Processing: Smoothing traces with gaussian filter with sigma: 1


Processing: Calculating baseline
Processing GP4_23_S1_L50_D250_mm2 - ('GP4-23', datetime.date(2022, 6, 26), 2)


Processing: Smoothing traces with gaussian filter with sigma: 1
Processing: Calculating baseline
Processing GP4_23_s3_L50_D250_M2 - ('GP4-23', datetime.date(2022, 6, 28), 1)


... (17 outputs left out)

Processing: Smoothing traces with gaussian filter with sigma: 1
Processing: Calculating baseline
Processing GP4_85_s8_L60_D250_ALM - ('GP4-85', datetime.date(2022, 7, 3), 2)


Processing: Smoothing traces with gaussian filter with sigma: 1
Processing: Calculating baseline
Processing GP4_85_s9_L70_D250_ALM - ('GP4-85', datetime.date(2022, 7, 4), 1)


Processing: Smoothing traces with gaussian filter with sigma: 1
Processing: Calculating baseline
Processing GP4_85_s10_L70_D250_ALM - ('GP4-85', datetime.date(2022, 7, 5), 1)


Processing: Collapsing the dataframes back into one dataframe


In [ ]:
def keepOnlyValidTraces(df, valid_long_traces_ids_set):
    valid_traces_by_sess = {} # session -> set of valid trace ids
    for long_trace_id in valid_long_traces_ids_set:
        sess_short_name, trace_id = long_trace_id.rsplit("_", 1)
        trace_id = int(trace_id)
        if sess_short_name not in valid_traces_by_sess:
            valid_traces_by_sess[sess_short_name] = set()
        valid_traces_by_sess[sess_short_name].add(trace_id)
    res_rows = []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        short_name = row.ShortName
        sess_valid_traces = valid_traces_by_sess.get(short_name)
        if sess_valid_traces is None: # Skip unprocessed brain-regions and layers
            continue
        new_traces_sets = {}
        for set_name, trace_dict in row.traces_sets.items():
            new_traces_dict = {trace_id:trace
                               for trace_id, trace in trace_dict.items()
                               if trace_id in sess_valid_traces}
            new_traces_sets[set_name] = new_traces_dict

        row = pipeline.updateRowTracesSet(row.copy(), new_traces_sets)
        res_rows.append(row)

    return pd.DataFrame(res_rows)

df_all_by_epoch_df_f_filtered = keepOnlyValidTraces(df_all_by_epoch_df_f,
                                                    set(is_active_unormed_df[is_active_unormed_df.valid].trace_id))

  0%|          | 0/22483 [00:00<?, ?it/s]

In [ ]:
from .twop.dataload import dataPath
from .util.portablepickle import savePortable

# Overwrites a file already under data/ (and in the download archive).
# savePortable refuses rather than writing one only this checkout can read.
if SAVE_DATA:
    _FP = dataPath("df_all_by_epoch_df_f_filtered.pkl")
    savePortable(df_all_by_epoch_df_f_filtered, _FP)


# Choice-selectivity tests (ROC), real and shuffled

Each neuron is tested for tuning to the current and previous trial's choice
and outcome, on the sampling epoch (`Unnormalized`) and on the feedback epoch
(`Feedback_fixed`). Run 0 is the real data, run 1 has the strategy labels
shuffled.

# Define generic function to run our tests

In [ ]:
from .twop.dimreduc import DimReduc
from .twop.statstest import StatsTest

defaultStatsTestFn = StatsTest.mannwhitneyu
defaultDimReducFn = DimReduc.max

cache_entry = f"{defaultDimReducFn('name')}_{defaultStatsTestFn('name',None)}"
if CONCAT_SAMPLING_MOVE:
  cache_entry += "_concat"
default_cache_dir = f"pkl/{cache_entry}"

print("Default Cache dir:", default_cache_dir)


Default Cache dir: pkl/max_MannWhitneyU_concat


# Run different conditions combinations

In [ ]:
from .twop.dataload import loadSamplingUnnormalized

df_rt_unnormalized = loadSamplingUnnormalized(num_quantiles=3, time_before=0.1,
                                              time_after=0.1)

# Run statistical tests on both real data and shuffled data

In [ ]:
from .twop.dataload import dataPath
from .twop.genrundata import (genRunDataInst, loadRunDataDict,
                                  saveRunDataDict)
from tqdm.auto import tqdm


import pickle
import pathlib

def genRandom(num_runs, df_src, fp_descrp):
    FP_PKL = str(dataPath(f"data_runs_{fp_descrp}.pkl"))
    if pathlib.Path(FP_PKL).exists():
        print("Loading old data from disk")
        run_data_dict = loadRunDataDict(FP_PKL)
        start = max(list(run_data_dict.keys())) if len(run_data_dict.keys()) \
                else -1
        start += 1
    else:
        print("Regenerting all data from scratch")
        run_data_dict = {}
        start = 0

    for run_idx in tqdm(range(start, num_runs+1)):
        if run_idx == 0:
            run_name = "Real"
            shuffle = False
        else:
            run_name = f"Rand{run_idx}"
            shuffle = True
        run_data_stats = genRunDataInst(df_src,
                                        shuffle_quantiles=shuffle,
                                        defaultDimReducFn=defaultDimReducFn,
                                        defaultStatsTestFn=defaultStatsTestFn,
                                        assign_name=run_name)
        run_data_dict[run_idx] = run_data_stats
        saveRunDataDict(run_data_dict, FP_PKL)
    return run_data_dict

In [ ]:
run_data_unnormalized = genRandom(num_runs=1, df_src=df_rt_unnormalized, fp_descrp="Unnormalized")

Loading old data from disk


0it [00:00, ?it/s]

In [ ]:
run_data_unnormalized[0].df_src.columns

Index(['Name', 'Date', 'SessionNum', 'TrialNumber', 'MaxTrial', 'MinSample',
       'calcStimulusTime', 'ChoiceCorrect', 'ChoiceLeft', 'EarlyWithdrawal',
       'PrevChoiceCorrect', 'PrevChoiceLeft', 'DV', 'DVstr', 'PrevDV',
       'PrevDVstr', 'Difficulty1', 'Difficulty2', 'Difficulty3', 'Difficulty4',
       'quantile_idx', 'ShortName', 'epoch', 'epoch_time', 'trace_start_idx',
       'trace_end_idx', 'traces_sets', 'sole_owner', 'acq_sampling_rate',
       'state_id', 'BrainRegion', 'Layer', 'Group'],
      dtype='object')

In [ ]:
import pandas as pd
from .twop.dataload import loadFeedbackTraces

df_feedback = loadFeedbackTraces()
df_feedback["epoch"] = "Feedback"
# Assign fast/typical/slow quantiles to the feedback trials
from .behavior.util.splitdata import splitStimulusTimeByQuantile
def assignQuantiles(df):
    df = df.copy()
    # Some files are saved twice due to cloud sync issues
    uniq_cols = ["Name", "Date", "SessionNum", "TrialNumber", "TrialStartSysTime"]
    df = df[~df[uniq_cols].duplicated(keep="first")]
    null_time = df.calcStimulusTime.isnull()
    df_null_time = df[null_time]
    df = df[~null_time]
    df = splitStimulusTimeByQuantile(df)
    df = pd.concat([q_df for q_idx, q_df in df] + [df_null_time])
    df = df.sort_values(by=["Name", "Date", "SessionNum", "TrialNumber", "epoch"])
    return df

df_feedback = assignQuantiles(df_feedback)
_cols = df_rt_unnormalized.columns.copy()
_cols = _cols.drop(['epochs_ranges', 'epochs_names'])
df_feedback = df_feedback[_cols]

  0%|          | 0/23 [00:00<?, ?it/s]

In [ ]:
run_data_feedback = genRandom(num_runs=1, df_src=df_feedback, fp_descrp="Feedback_fixed")

Loading old data from disk


0it [00:00, ?it/s]

In [ ]:
run_data_feedback[0].df_src

,Name,Date,SessionNum,TrialNumber,MaxTrial,MinSample,calcStimulusTime,ChoiceCorrect,ChoiceLeft,EarlyWithdrawal,...,trace_end_idx,traces_sets,sole_owner,acq_sampling_rate,state_id,SamplingType,BrainRegion,Layer,PrevOutcomeCount,Group
19229,GP4-23,2022-06-26,2,42,333,0.3,1.1627,1.0,1.0,0,...,83,"{'neuronal': {1: [7168.2236, 6967.162, 7056.62...",False,30.043,19,ReactionTime,15,L23,0,Real
19233,GP4-23,2022-06-26,2,43,333,0.3,0.8047,1.0,0.0,0,...,235,"{'neuronal': {1: [7168.2236, 6967.162, 7056.62...",False,30.043,19,ReactionTime,15,L23,1,Real
19237,GP4-23,2022-06-26,2,44,333,0.3,0.8356,1.0,0.0,0,...,338,"{'neuronal': {1: [7168.2236, 6967.162, 7056.62...",False,30.043,19,ReactionTime,15,L23,2,Real
19241,GP4-23,2022-06-26,2,45,333,0.3,0.7941,0.0,0.0,0,...,469,"{'neuronal': {1: [7168.2236, 6967.162, 7056.62...",False,30.043,25,ReactionTime,15,L23,3,Real
19245,GP4-23,2022-06-26,2,46,333,0.3,0.8260,1.0,0.0,0,...,632,"{'neuronal': {1: [7168.2236, 6967.162, 7056.62...",False,30.043,19,ReactionTime,15,L23,-1,Real
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9897,GP4-85,2022-07-05,1,318,327,0.3,0.6534,1.0,0.0,0,...,35082,"{'neuronal': {0: [600.8341, 548.72266, 567.048...",False,30.043,19,ReactionTime,6,L23,3,Real
9901,GP4-85,2022-07-05,1,319,327,0.3,0.8686,1.0,0.0,0,...,35178,"{'neuronal': {0: [600.8341, 548.72266, 567.048...",False,30.043,19,ReactionTime,6,L23,4,Real
9905,GP4-85,2022-07-05,1,320,327,0.3,0.9118,1.0,1.0,0,...,35287,"{'neuronal': {0: [600.8341, 548.72266, 567.048...",False,30.043,19,ReactionTime,6,L23,5,Real
9909,GP4-85,2022-07-05,1,321,327,0.3,0.7944,1.0,0.0,0,...,35343,"{'neuronal': {0: [600.8341, 548.72266, 567.048...",False,30.043,19,ReactionTime,6,L23,6,Real


# Save sgf. neurons to disk

In [ ]:
from .twop.dataload import dataPath

sgf_all = run_data_unnormalized[0].shortlong_df.query(f'pval <= {PVAL} and DVstr.isnull()')
if SAVE_DATA:
    with open(dataPath("sgf_all.pkl"), "wb") as f:
        pickle.dump(sgf_all, f)
sgf_all

,trace_id,data_col,data_val_left,data_val_right,prior_data_col,prior_data_val,ShortName,dim_reduc_name,stat_test_name,left_mean,...,BrainRegion,Layer,Group,long_trace_id,left_mean_roc,right_mean_roc,left_median_roc,right_median_roc,statistic_roc,IsROCLeftTuend
4338,1,ChoiceLeft,True,False,None,NaN,GP4_23_S1_L50_D250_mm2,max,MannWhitneyU,5.636729,...,15,L23,Real,GP4_23_S1_L50_D250_mm2_1,3.059228,1.742934,1.838211,0.537988,0.676847,True
4339,2,ChoiceLeft,True,False,None,NaN,GP4_23_S1_L50_D250_mm2,max,MannWhitneyU,2.853587,...,15,L23,Real,GP4_23_S1_L50_D250_mm2_2,1.441019,2.438052,0.376516,1.152115,0.408320,False
4341,4,ChoiceLeft,True,False,None,NaN,GP4_23_S1_L50_D250_mm2,max,MannWhitneyU,1.160029,...,15,L23,Real,GP4_23_S1_L50_D250_mm2_4,0.538118,0.388174,0.309725,0.244734,0.582430,True
4344,12,ChoiceLeft,True,False,None,NaN,GP4_23_S1_L50_D250_mm2,max,MannWhitneyU,1.837721,...,15,L23,Real,GP4_23_S1_L50_D250_mm2_12,0.852148,1.047164,0.401155,0.533479,0.397263,False
4349,17,ChoiceLeft,True,False,None,NaN,GP4_23_S1_L50_D250_mm2,max,MannWhitneyU,1.513813,...,15,L23,Real,GP4_23_S1_L50_D250_mm2_17,0.739895,1.189286,0.355784,0.757591,0.402189,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14449,46,PrevChoiceCorrect,True,False,None,NaN,GP4_85_s10_L70_D250_ALM,max,MannWhitneyU,1.082674,...,6,L23,Real,GP4_85_s10_L70_D250_ALM_46,0.462193,0.208853,0.216210,0.134313,0.659091,True
14451,53,PrevChoiceCorrect,True,False,None,NaN,GP4_85_s10_L70_D250_ALM,max,MannWhitneyU,1.065751,...,6,L23,Real,GP4_85_s10_L70_D250_ALM_53,0.401103,0.652263,0.339345,0.434918,0.282529,False
14453,58,PrevChoiceCorrect,True,False,None,NaN,GP4_85_s10_L70_D250_ALM,max,MannWhitneyU,1.141320,...,6,L23,Real,GP4_85_s10_L70_D250_ALM_58,0.432175,0.317065,0.298357,0.263263,0.568397,True
14455,62,PrevChoiceCorrect,True,False,None,NaN,GP4_85_s10_L70_D250_ALM,max,MannWhitneyU,0.761462,...,6,L23,Real,GP4_85_s10_L70_D250_ALM_62,0.313703,0.200107,0.196079,0.119358,0.653813,True


# Max-firing tables

Each neuron's peak sample on every trial, split into fast / typical / slow
trials. Rebuilding takes ~20 minutes, so the tables are read from `data/2p`
unless `LOAD_FROM_PICKLE` is set to False below.

In [ ]:
from .twop.dataload import loadSamplingNormalized

normed_sampling_df = loadSamplingNormalized(time_before=0.1, time_after=0.1)

In [ ]:
BINS_COUNT = 7

#### Assign quantiles if we don't have quantiles

In [ ]:
from .twop.samplingquantiles import assignQuantileIdx

normed_sampling_df = assignQuantileIdx(normed_sampling_df)

In [ ]:
# The trace-reliability helpers are tested in twop/tracereliability.py.
from .twop.dataload import dataPath, loadMaxFiring
from .twop.tracereliability import (DecideNeurons, StdDistCollector,
                                    loopSessionsExampleNeurons)
import numpy as np
import pickle

# The four dicts below take ~20 minutes to build, so they are cached under
# data/2p/. Set this False to rebuild them (and re-save them below). The cached
# copies carry neither the ChoiceCorrect nor the std_threshold column; a rebuild
# adds both, and nothing downstream reads them.
LOAD_FROM_PICKLE = True
if LOAD_FROM_PICKLE:
    print("Loading from pickle...")
    first_q_res_dict = loadMaxFiring("fast")
    second_q_res_dict = loadMaxFiring("typical")
    thrd_q_res_dict = loadMaxFiring("slow")
    all_res_normed_dict = loadMaxFiring("all")
else:
    np.seterr(all='raise')
    _decide_neurons = DecideNeurons(smooth_sigma=1)

    def _collect(df_query=""):
        collector = StdDistCollector(_decide_neurons)
        loopSessionsExampleNeurons(normed_sampling_df, random_fraction=1, seed=1,
                                   pdf_str="max_firing", df_query=df_query,
                                   processFn=collector.track, save_figs=False)
        return collector.res_dict

    all_res_normed_dict = _collect()
    first_q_res_dict = _collect("quantile_idx == 1")
    second_q_res_dict = _collect("quantile_idx == 2")
    thrd_q_res_dict = _collect("quantile_idx == 3")

    if SAVE_DATA:
        for _dict, save_fp in [(first_q_res_dict, dataPath("q1_res_dict.pkl")),
                               (second_q_res_dict, dataPath("q2_res_dict.pkl")),
                               (thrd_q_res_dict, dataPath("q3_res_dict.pkl")),
                               (all_res_normed_dict, dataPath("qall_res_dict.pkl"))]:
            with open(save_fp, 'wb') as f:
                pickle.dump(_dict, f)

Loading from pickle...
